# Caribu for developers, 0: overview of the package

**For:** developers who want to read, check, change or couple the light model. **You need:** the user tutorial
`../user/01_first_run.ipynb` (what the model computes), Python, numpy. **Run time:** a few seconds.

This series:

| | notebook | what it covers |
|---|---|---|
| 0 | `00_overview.ipynb` | the package's modules, the original Caribu, how the port is validated, where the equations are |
| 1 | `01_caribu_in_metafspm.ipynb` | metafspm's component formalism on `CaribuLight`, the light models on triangles, the light sources, the infinite canopy and the corrections, the `Canopy`, `CaribuLight` in a scene of Adel's wheat, precision and speed |
| 2 | `02_extend_the_component.ipynb` | add a variable and a step to the component, light a scene with it, test it |

## 1. Where the model comes from

**Caribu** (Chelle et al., 1998, and its later developments at INRAE) is a suite of light models for plant canopies:
a projection model (z-buffer) for the first order, radiosity for the exchanges between nearby triangles, and a turbid
medium (SAIL) for the rest of an infinite canopy, combined in the **nested radiosity**. Its original form is C++
executables (canestrad, s2v, mcsail, periodise: the package `openalea.libcaribu`) driven by Python through files
(`openalea.caribu.legacy`, kept frozen).

This package is Caribu ported to Python (numpy, numba), without the executables or files, and a metafspm component,
`CaribuLight`, built on it. The docs' **legacy** page (`doc/legacy.rst`) describes the original chain, its algorithms
and every quirk found, each with what the port does; the **equations** page (`doc/equations.rst`) the models'
equations and the functions computing them; the **performance** page (`doc/performance.rst`) the precision, the
corrections and the speed.

## 2. The modules

Each module's first docstring line (`legacy` is the original package, `comparison` drives it for the tests):

In [1]:
import importlib
import pkgutil

import openalea.caribu as package

for module in pkgutil.iter_modules(package.__path__):
    if module.name == "legacy":
        continue
    doc = (importlib.import_module(f"openalea.caribu.{module.name}").__doc__ or "").strip()
    print(f"{module.name:12} {doc.splitlines()[0] if doc else ''}")

api          The legacy's entry points over the port: `raycasting`, `x_raycasting` (direct light), `radiosity` and `x_radiosity`
budgets      From radiosities to the triangles' light budgets (canestrad's genres; doc/legacy.rst). A face of reflectance
canopy       A static canopy prepared for repeated lighting (a FunctionalComponent's state): what depends only on its geometry
comparison   The port against the legacy: libcaribu's executables run on a scene (legacy_run) and the port on the same one
component    Caribu as a metafspm component: the light each element of a canopy absorbs, from its triangles.
corrections  The corrections of the legacy's behaviours (doc/legacy.rst's quirk registry): each a name, applied when given in a model's
data         
direct       The first-order radiosities: each face's light from the sources only, before any scattering (canestrad's main light
formats      Caribu's file formats, read and written: the scene (`.can`: one triangle per line, `p 1 <label> 3 

- `direct.py`, `nested.py`, `radiosity.py`, `sail.py` are the light models on arrays of triangles; `projection.py`,
  `nearfield.py`, `periodic.py`, `solver.py`, `geometry.py` their parts; `gpu.py` the near field on a GPU.
- `canopy.py` keeps what depends only on the geometry between calls (`Canopy`).
- `lights.py` builds the light sources (skies, the sun, the diffuse fraction).
- `api.py` is the original Python interface (`CaribuScene`, `raycasting`, ...) on the port.
- `corrections.py` names the corrections of the original's behaviours.
- `component.py` is the metafspm component `CaribuLight` and its environment model `CaribuLightModel`.
- `simulation.py` is the users' entry point: `simulate`, a stand of `SimplePlant`s, the tables.

## 3. How the port is validated

The tests, in `test/`, run with `pytest test` from the repository's root. Each file checks one aspect:

In [2]:
import ast
from pathlib import Path

for folder in ("../../../test", "../../../test/comparison"):
    for path in sorted(Path(folder).glob("test_*.py")):
        doc = ast.get_docstring(ast.parse(path.read_text())) or ""
        name = f"{Path(folder).name}/{path.name}" if folder.endswith("comparison") else path.name
        print(f"{name:34} {' '.join(doc.split())[:120]}")

test_api.py                        The port's API beyond the legacy's: full precision, bands with a soil (a TypeError in the legacy).
test_canopy.py                     Canopy: repeated lighting of a static scene from its caches gives the results of fresh runs.
test_component.py                  CaribuLight, the metafspm component: on a population's DataStructure and on a union of two in a Scene, against CaribuSce
test_component_adel.py             CaribuLight on Adel's port (ShootArchitecture's triangles): a stand grown in a metafspm Scene, lit every 4 hours, agains
test_corrections.py                The corrections of the legacy's behaviours (C10): each restores a physical property the legacy misses, checked on small 
test_direct.py                     Direct light on cases with a known answer.
test_formats.py                    The port's file writers give the legacy's text (caribu.py's builders), its readers read it back.
test_gpu.py                        The near field's CUDA ker

Three levels:

- **Exactness.** With `precision="legacy"` and no corrections, every output equals the original's to its printed
  digits. `test/comparison` runs the original executables (from `openalea.libcaribu`) on the same scenes and compares
  their files and the API's dicts; the original package's own tests run on the port (`test_legacy_suite.py`).
  `CaribuLight` with these options reproduces Wheat-BRIDGES' use of Caribu exactly (`test_component_adel.py`).
- **Corrections.** Each behaviour of the original found to be a defect is corrected by a named correction
  (`openalea.caribu.corrections`), tested by the physics it restores (`test_corrections.py`). `"recommended"`, the
  component's default, applies them all.
- **The component and the users' entry point** are tested against the models underneath them (`test_component.py`,
  `test_simulation.py`).

The corrections, by name:

In [3]:
from openalea.caribu import corrections

for name, description in corrections.NAMES.items():
    print(f"{name:>22}  {description}")

         border_copies  a triangle across the pattern's borders copied on each side it crosses, corners included
                tiling  the neighbouring patterns' images placed by the lattice's exact vectors, each translate rounded on its own, its depth offset exact
         sensor_copies  sensors copied across the borders as triangles, each copy's pixels counted once
            pixel_area  a pixel's area the pitch's, du dv / (T - 1)^2
          upward_beams  a source whose beam goes up or along the ground lights nothing
             periodise  the periodised scene in float32 without its text rounding, lifted to exactly z >= 0
         triangle_area  a triangle's area 0.5 |AB x AC|
           voxel_floor  the receiver's voxel floor((G - O) / side)
           face_counts  an emitter's two faces counted apart
        projected_face  each pixel's emitter face the one of the copy projected there
 profile_interpolation  the far field's levels interpolated with weights (1 - f, f)
        p

## 4. What to read next

- `01_caribu_in_metafspm.ipynb` reads the component's code through metafspm's component formalism, then goes down
  to the light models, the sources, the infinite canopy and the corrections, and up to a scene of growing wheat.
- `02_extend_the_component.ipynb` adds a variable and a step, and lights a scene with them.
- metafspm's own tutorial (`examples/soil_plant_atmosphere/tutorial_fspm2026_general_usage.ipynb` in the metafspm
  repository) builds components and couples them step by step; its user guide, `docs/user.md`, describes every part
  of the framework.